# Use A/A assignments to analyze orders in an A/B test

[All How-to Guides](README.md)

Save a stratified A/A assignment, use it to simulate an order log, and build a table with one row per assigned user. Include users who placed no orders.

## 1. Assign users using historical data

The strata and metric are available before the experiment. `best_split` holds each user's assignment. Keep the user ID and assignment for the next steps.

`stratification=True` splits within segments marked with `StratificationRole`. The three seeds in `random_states` give three reproducible candidate splits. The default control share is 0.5.

Use HypEx to select and rename columns. The public `data` property gives pandas the table it needs to build and join the synthetic event log.

In [1]:
import numpy as np
import pandas as pd

from hypex import AATest, ABTest
from hypex.dataset import (
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)

rng = np.random.default_rng(73)
n_users = 600
history = pd.DataFrame(
    {
        "user_id": np.arange(n_users),
        "segment": rng.choice(["new", "returning"], n_users),
        "pre_revenue": rng.gamma(4, 10, n_users),
    }
)
historical = Dataset(
    roles={
        "user_id": InfoRole(int),
        "segment": StratificationRole(str),
        "pre_revenue": TargetRole(),
    },
    data=history,
)
split = AATest(stratification=True, random_states=[10, 11, 12]).execute(historical)
assignment = (
    split.best_split.select(["user_id", "split"]).rename({"split": "variant"}).data
)
{
    "unique users": assignment["user_id"].nunique(),
    "groups": assignment["variant"].value_counts().to_dict(),
}

{'unique users': 600, 'groups': {'control': 306, 'test_1': 294}}

Each user ID belongs to one group. Save this table for serving variants and analyzing the experiment.

## 2. Build the metric from events

Simulate orders during the experiment window. Add some duplicate records to exercise deduplication. Aggregate unique orders by user and join the totals to the complete assignment table.

In [2]:
order_users = np.repeat(
    assignment["user_id"].to_numpy(), rng.poisson(1.0, size=len(assignment))
)
is_test = (
    pd.Series(order_users).map(assignment.set_index("user_id")["variant"]).eq("test_1")
)
orders = pd.DataFrame(
    {
        "order_id": np.arange(len(order_users)),
        "user_id": order_users,
        "amount": rng.gamma(4, 10, size=len(order_users)) + 6 * is_test,
    }
)
events = pd.concat([orders, orders.head(8)], ignore_index=True)
unique_orders = events.drop_duplicates("order_id")
revenue = unique_orders.groupby("user_id")["amount"].sum()
cohort = assignment.join(revenue.rename("revenue"), on="user_id", validate="one_to_one")
cohort["revenue"] = cohort["revenue"].fillna(0.0)
{
    "unique orders": len(unique_orders),
    "eligible users": len(cohort),
    "users with zero revenue": int(cohort["revenue"].eq(0).sum()),
}

{'unique orders': 583, 'eligible users': 600, 'users with zero revenue': 230}

After deduplication, 583 orders remain. The analysis includes all 600 users, 230 of whom have zero revenue because they placed no order during the simulated window. In production event logs, check that records sharing an `order_id` have identical contents before dropping duplicates. The example creates duplicates by copying records.

## 3. Run the A/B test on the saved groups

Assign `TreatmentRole` to `variant` and `TargetRole` to total revenue in the experiment window. The outcome is revenue per assigned user.

`equal_variance=False` explicitly selects Welch's t-test, the HypEx default. The group assignments come from `TreatmentRole` with either variance setting.

In [3]:
readout = Dataset(
    roles={
        "user_id": InfoRole(int),
        "variant": TreatmentRole(str),
        "revenue": TargetRole(),
    },
    data=cohort,
)
ABTest(equal_variance=False).execute(readout).summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,revenue,test_1,38.932831,44.442836,5.510005,14.152593,-2.045328,13.065338,NOT OK,0.153411


The difference in means is about 5.51 units of revenue per assigned user. The interval from −2.05 to 13.07 includes zero. For production events, use the experiment window and compute one outcome per unit of assignment.